<center><h1 style="margin: 1.5em 0;">Mathematical Explanation of the MNISTNet Implementation</h1></center>



A 3-layer fully connected neural network (784 → 256 → 128 → 10) trained with **mini-batch gradient descent**, **ReLU** hidden activations, **softmax** output, **cross-entropy loss** + **L2 regularization**.

## 1. Notation

- $X \in \mathbb{R}^{m \times 784}$ : mini-batch of flattened images  
- $Y \in \mathbb{R}^{m \times 10}$ : one-hot encoded true labels  
- $m$ : batch size  
- $W^{(l)} \in \mathbb{R}^{n_l \times n_{l-1}}$ : weight matrix of layer $l$  
- $b^{(l)} \in \mathbb{R}^{1 \times n_l}$ : bias vector  
- $Z^{(l)}, A^{(l)} \in \mathbb{R}^{m \times n_l}$ : pre- and post-activation of layer $l$  
- $A^{(0)} := X$

Layers:  
- $l=1$: 784 → 256  
- $l=2$: 256 → 128  
- $l=3$: 128 → 10 (output)

## 2. Forward Pass

$$
\begin{aligned}
Z^{(1)} &= X W^{(1)\top} + b^{(1)} \\
A^{(1)} &= \operatorname{ReLU}(Z^{(1)}) && \text{where } \operatorname{ReLU}(z) = \max(0,z) \\[1em]
Z^{(2)} &= A^{(1)} W^{(2)\top} + b^{(2)} \\
A^{(2)} &= \operatorname{ReLU}(Z^{(2)}) \\[1em]
Z^{(3)} &= A^{(2)} W^{(3)\top} + b^{(3)} \\
A^{(3)} &= \operatorname{softmax}(Z^{(3)})
\end{aligned}
$$

**Stable softmax** (used in code):

$$
\operatorname{softmax}(z_i) = \frac{\exp(z_i - \max_j z_j)}{\sum_k \exp(z_k - \max_j z_j)}
$$

$A^{(3)}$ contains predicted probabilities $\hat{y}_i \in [0,1]^{10}$ with $\sum_c \hat{y}_{i,c} = 1$.

## 3. Loss Function

**Per mini-batch loss** (average cross-entropy + L2 regularization):

$$
\mathcal{L} = -\frac{1}{m} \sum_{i=1}^m \sum_{c=1}^{10} y_{i,c} \log(\hat{y}_{i,c}) + \frac{\lambda}{2m} \sum_{l=1}^3 \|W^{(l)}\|_F^2
$$

- First term = categorical cross-entropy (CE)  
- Second term = weight decay (L2 regularization)  
- $\lambda$ = regularization strength (`reg` in code)

## 4. Backpropagation – Gradient Computation

### Output layer (l = 3)

$$
\frac{\partial \mathcal{L}}{\partial Z^{(3)}} = A^{(3)} - Y \qquad \text{(very convenient property of softmax + CE)}
$$

$$
\frac{\partial \mathcal{L}}{\partial W^{(3)}} = \frac{1}{m} \left( {dZ^{(3)}}^\top A^{(2)} \right) + \frac{\lambda}{m} W^{(3)}
$$

$$
\frac{\partial \mathcal{L}}{\partial b^{(3)}} = \frac{1}{m} \sum_{i=1}^m dZ^{(3)}_{i,:} \quad \text{(row vector)}
$$

### Hidden layers (l = 2 and l = 1)

$$
dZ^{(l)} = \left( dZ^{(l+1)} W^{(l+1)} \right) \odot \mathbb{1}_{\{Z^{(l)} > 0\}}
$$

$$
\frac{\partial \mathcal{L}}{\partial W^{(l)}} = \frac{1}{m} \left( {dZ^{(l)}}^\top A^{(l-1)} \right) + \frac{\lambda}{m} W^{(l)}
$$

$$
\frac{\partial \mathcal{L}}{\partial b^{(l)}} = \frac{1}{m} \sum_{i=1}^m dZ^{(l)}_{i,:}
$$

(where $A^{(0)} = X$ and $\odot$ is element-wise multiplication)

## 5. Parameter Update (Vanilla Gradient Descent)

$$
W^{(l)} \leftarrow W^{(l)} - \eta \cdot \frac{\partial \mathcal{L}}{\partial W^{(l)}}, \quad
b^{(l)} \leftarrow b^{(l)} - \eta \cdot \frac{\partial \mathcal{L}}{\partial b^{(l)}}
$$

$\eta =$ learning rate (`lr` in code)

## 6. Weight Initialization – He initialization (for ReLU)

$$
W^{(l)}_{ij} \sim \mathcal{N}\left(0, \sqrt{\frac{2}{n_{l-1}}}\right)
$$

This scaling helps keep the variance of activations roughly constant across layers when using ReLU.

## Summary Table

| Layer | Input dim | Output dim | Activation | Gradient w.r.t. Z          |
|-------|-----------|------------|------------|-----------------------------|
| 1     | 784       | 256        | ReLU       | $(dZ^{(2)} W^{(2)}) \odot \mathbb{1}_{Z^{(1)}>0}$ |
| 2     | 256       | 128        | ReLU       | $(dZ^{(3)} W^{(3)}) \odot \mathbb{1}_{Z^{(2)}>0}$ |
| 3     | 128       | 10         | softmax    | $A^{(3)} - Y$               |



## Gradient w.r.t. Weights: ∂L/∂W³ , ∂L/∂W² , ∂L/∂W¹

These are the gradients of the total loss L with respect to each weight matrix.  
They tell us **in which direction and by how much we should change each W** to reduce the loss.

### General pattern (same for every layer)

For any fully-connected layer l:

$$
\frac{\partial L}{\partial \mathbf{W}^{(l)}}
= \frac{1}{m} \left( {\delta^{(l)}}^\top \, \mathbf{A}^{(l-1)} \right)
+ \frac{\lambda}{m} \, \mathbf{W}^{(l)}
$$

- First term = **data gradient** (how much the weights affect the loss through the current batch)
- Second term = **L₂ regularization gradient** (weight decay / decay toward zero)

In the code this appears as:

```python
grads["W3"] = (dZ3.T @ A2) / m + (self.reg / m) * W3
grads["W2"] = (dZ2.T @ A1) / m + (self.reg / m) * W2
grads["W1"] = (dZ1.T @ X)  / m + (self.reg / m) * W1

## Layer-by-Layer Breakdown  
**How the gradients ∂L/∂W³, ∂L/∂W², and ∂L/∂W¹ are computed**

These three gradients tell the network **exactly how to update each weight matrix** to reduce the loss.

### General Formula (same for every layer)

$$
\frac{\partial L}{\partial \mathbf{W}^{(l)}}
= \frac{1}{m} \Bigl( {\boldsymbol{\delta}^{(l)}}^\top \, \mathbf{A}^{(l-1)} \Bigr)
+ \frac{\lambda}{m} \, \mathbf{W}^{(l)}
$$

- First term → **data gradient** (what the current batch is telling us)  
- Second term → **L₂ regularization** (weight decay)

---

### 1. Output Layer → ∂L / ∂W³

Error signal (very simple because of softmax + cross-entropy):

$$
\boldsymbol{\delta}^{(3)} = \hat{\mathbf{Y}} - \mathbf{Y}
\quad \in \mathbb{R}^{m \times 10}
$$

Gradient:

$$
\frac{\partial L}{\partial \mathbf{W}^{(3)}}
= \frac{1}{m} \bigl( {\boldsymbol{\delta}^{(3)}}^\top \, \mathbf{A}^{(2)} \bigr)
+ \frac{\lambda}{m} \, \mathbf{W}^{(3)}
$$

**Shape check**  
δ³ᵀ (10×m) @ A² (m×128) → (10×128) → exactly shape of W³

**Intuition**  
Each row of this gradient tells us:  
“How should the 128 incoming weights to output class j be changed so that class j becomes more (or less) confident?”

---

### 2. Second Hidden Layer → ∂L / ∂W²

First compute the error signal for this layer:

$$
\boldsymbol{\delta}^{(2)} = \bigl( \boldsymbol{\delta}^{(3)} \, \mathbf{W}^{(3)} \bigr)
\odot \operatorname{ReLU}'(\mathbf{Z}^{(2)})
\quad \in \mathbb{R}^{m \times 128}
$$

Gradient:

$$
\frac{\partial L}{\partial \mathbf{W}^{(2)}}
= \frac{1}{m} \bigl( {\boldsymbol{\delta}^{(2)}}^\top \, \mathbf{A}^{(1)} \bigr)
+ \frac{\lambda}{m} \, \mathbf{W}^{(2)}
$$

**Shape**: (128 × 256) — matches W²

**Intuition**  
δ² tells us “how much each of the 128 hidden neurons contributed to the final mistake”.  
Multiplying by A¹ gives the correlation between those neurons and the features coming from the first hidden layer.

---

### 3. First Hidden Layer → ∂L / ∂W¹

Error signal:

$$
\boldsymbol{\delta}^{(1)} = \bigl( \boldsymbol{\delta}^{(2)} \, \mathbf{W}^{(2)} \bigr)
\odot \operatorname{ReLU}'(\mathbf{Z}^{(1)})
\quad \in \mathbb{R}^{m \times 256}
$$

Gradient:

$$
\frac{\partial L}{\partial \mathbf{W}^{(1)}}
= \frac{1}{m} \bigl( {\boldsymbol{\delta}^{(1)}}^\top \, \mathbf{X} \bigr)
+ \frac{\lambda}{m} \, \mathbf{W}^{(1)}
$$

**Shape**: (256 × 784) — matches W¹

**Intuition**  
This gradient shows which connections from the 784 raw pixels to the first hidden layer are most responsible for the network’s current mistakes.

---

### Summary Table

| Weight | Shape       | Error signal δ used                          | Previous activation | Code line in `backward()`                     | Role in learning                              |
|--------|-------------|----------------------------------------------|---------------------|-----------------------------------------------|-----------------------------------------------|
| W³     | 10×128      | δ³ = Ŷ − Y                                   | A² (128-dim)        | `(dZ3.T @ A2)/m + (reg/m)*W3`                | Final class confidence                        |
| W2     | 128×256     | δ² = (δ³ W³) ⊙ ReLU'(Z²)                     | A¹ (256-dim)        | `(dZ2.T @ A1)/m + (reg/m)*W2`                | Mid-level feature refinement                  |
| W1     | 256×784     | δ¹ = (δ² W2) ⊙ ReLU'(Z¹)                     | X (pixels)          | `(dZ1.T @ X)/m  + (reg/m)*W1`                | Low-level pattern detection (edges, strokes)  |

### One-sentence takeaway for each gradient

- **∂L/∂W³** → tells us how to adjust the final class weights to make correct digits more likely and wrong digits less likely  
- **∂L/∂W²** → tells us how to improve the second hidden layer’s features for the output  
- **∂L/∂W¹** → tells us which pixel patterns are most useful for distinguishing handwritten digits

All three gradients are used in exactly the same way:

```python
self.params["W3"] -= self.lr * grads["W3"]
self.params["W2"] -= self.lr * grads["W2"]
self.params["W1"] -= self.lr * grads["W1"]